In [13]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import linregress

print("Libraries loaded successfully")

Libraries loaded successfully


In [14]:
# NAV History
nav = pd.read_csv("../data/raw/02_nav_history.csv")

# Scheme Performance
performance = pd.read_csv("../data/raw/07_scheme_performance.csv")

# Benchmark Data
benchmark = pd.read_csv("../data/raw/10_benchmark_indices.csv")

# Fund Master
funds = pd.read_csv("../data/raw/01_fund_master.csv")

print("NAV:", nav.shape)
print("Performance:", performance.shape)
print("Benchmark:", benchmark.shape)
print("Funds:", funds.shape)

NAV: (46000, 3)
Performance: (40, 19)
Benchmark: (8050, 3)
Funds: (40, 15)


In [15]:
# Convert date column

nav["date"] = pd.to_datetime(nav["date"])

# Sort values

nav = nav.sort_values(["amfi_code", "date"])

# Daily Return

nav["daily_return"] = nav.groupby("amfi_code")["nav"].pct_change()

# Check result

nav.head()

,amfi_code,date,nav,daily_return
5750,100016,2022-01-03,520.4608,NaN
5751,100016,2022-01-04,515.0971,-0.010306
5752,100016,2022-01-05,521.7239,0.012865
5753,100016,2022-01-06,515.7880,-0.011377
5754,100016,2022-01-07,515.1639,-0.001210


In [16]:
print(nav[["amfi_code","date","nav","daily_return"]].head(10))

      amfi_code       date       nav  daily_return
5750     100016 2022-01-03  520.4608           NaN
5751     100016 2022-01-04  515.0971     -0.010306
5752     100016 2022-01-05  521.7239      0.012865
5753     100016 2022-01-06  515.7880     -0.011377
5754     100016 2022-01-07  515.1639     -0.001210
5755     100016 2022-01-10  510.7136     -0.008639
5756     100016 2022-01-11  513.5542      0.005562
5757     100016 2022-01-12  512.3195     -0.002404
5758     100016 2022-01-13  510.2445     -0.004050
5759     100016 2022-01-14  514.3636      0.008073


In [17]:
# CAGR Calculation

cagr_results = []

for fund in nav["amfi_code"].unique():

    fund_data = nav[nav["amfi_code"] == fund].sort_values("date")

    start_nav = fund_data["nav"].iloc[0]
    end_nav = fund_data["nav"].iloc[-1]

    years = (
        (fund_data["date"].iloc[-1] -
         fund_data["date"].iloc[0]).days
    ) / 365

    cagr = ((end_nav / start_nav) ** (1 / years) - 1) * 100

    cagr_results.append([fund, cagr])

cagr_df = pd.DataFrame(
    cagr_results,
    columns=["amfi_code", "CAGR"]
)

cagr_df.head()

,amfi_code,CAGR
0,100016,2.635246
1,100025,4.455091
2,100033,30.099704
3,101206,23.520489
4,101207,7.933121


In [18]:
cagr_df = cagr_df.sort_values(
    "CAGR",
    ascending=False
)

print(cagr_df.head(10))

    amfi_code       CAGR
25     120505  32.801599
21     119598  32.398084
39     149324  32.262108
36     148569  31.924486
34     148567  30.949920
30     120843  30.883326
2      100033  30.099704
38     149323  29.558105
16     119094  28.192608
19     119551  25.784921


In [19]:
risk_free_rate = 0.065

sharpe_results = []

for fund in nav["amfi_code"].unique():

    fund_returns = nav[
        nav["amfi_code"] == fund
    ]["daily_return"].dropna()

    avg_return = fund_returns.mean() * 252

    volatility = fund_returns.std() * np.sqrt(252)

    sharpe = (avg_return - risk_free_rate) / volatility

    sharpe_results.append([fund, sharpe])

sharpe_df = pd.DataFrame(
    sharpe_results,
    columns=["amfi_code", "Sharpe_Ratio"]
)

sharpe_df.head()

,amfi_code,Sharpe_Ratio
0,100016,-0.201517
1,100025,-0.567095
2,100033,1.093699
3,101206,1.027213
4,101207,0.162661


In [20]:
sharpe_df = sharpe_df.sort_values(
    "Sharpe_Ratio",
    ascending=False
)

print(sharpe_df.head(10))

    amfi_code  Sharpe_Ratio
34     148567      1.448291
30     120843      1.306744
36     148569      1.234930
19     119551      1.208267
25     120505      1.180101
38     149323      1.132122
2      100033      1.093699
9      118632      1.081659
3      101206      1.027213
24     120504      1.026524


In [21]:
risk_free_rate = 0.065

sortino_results = []

for fund in nav["amfi_code"].unique():

    fund_returns = nav[
        nav["amfi_code"] == fund
    ]["daily_return"].dropna()

    avg_return = fund_returns.mean() * 252

    downside_returns = fund_returns[
        fund_returns < 0
    ]

    downside_std = downside_returns.std() * np.sqrt(252)

    sortino = (
        avg_return - risk_free_rate
    ) / downside_std

    sortino_results.append(
        [fund, sortino]
    )

sortino_df = pd.DataFrame(
    sortino_results,
    columns=["amfi_code", "Sortino_Ratio"]
)

sortino_df.head()

,amfi_code,Sortino_Ratio
0,100016,-0.351047
1,100025,-0.941821
2,100033,1.829134
3,101206,1.799563
4,101207,0.276644


In [22]:
sortino_df = sortino_df.sort_values(
    "Sortino_Ratio",
    ascending=False
)

print(sortino_df.head(10))

    amfi_code  Sortino_Ratio
34     148567       2.385644
30     120843       2.364320
36     148569       2.146914
19     119551       2.140267
25     120505       2.029353
38     149323       1.875101
9      118632       1.850133
2      100033       1.829134
24     120504       1.805294
3      101206       1.799563


In [23]:
print(benchmark.columns)
benchmark.head()

Index(['date', 'index_name', 'close_value'], dtype='str')


,date,index_name,close_value
0,2022-01-03,NIFTY50,17492.79
1,2022-01-04,NIFTY50,17689.64
2,2022-01-05,NIFTY50,17835.05
3,2022-01-06,NIFTY50,17878.51
4,2022-01-07,NIFTY50,17759.15


In [24]:
# Benchmark returns

benchmark["date"] = pd.to_datetime(benchmark["date"])

benchmark = benchmark.sort_values("date")

benchmark["benchmark_return"] = benchmark["close_value"].pct_change()

benchmark.head()

,date,index_name,close_value,benchmark_return
0,2022-01-03,NIFTY50,17492.79,NaN
5750,2022-01-03,CRISIL_LIQUID,2281.51,-0.869574
2300,2022-01-03,NIFTY_MIDCAP150,9721.79,3.261121
6900,2022-01-03,CRISIL_GILT,1451.06,-0.850741
1150,2022-01-03,NIFTY100,17778.24,11.251899


In [25]:
alpha_beta_results = []

for fund in nav["amfi_code"].unique():

    fund_data = nav[
        nav["amfi_code"] == fund
    ][["date", "daily_return"]]

    merged = pd.merge(
        fund_data,
        benchmark[["date", "benchmark_return"]],
        on="date",
        how="inner"
    )

    merged = merged.dropna()

    if len(merged) > 30:

        slope, intercept, r_value, p_value, std_err = linregress(
            merged["benchmark_return"],
            merged["daily_return"]
        )

        alpha = intercept * 252
        beta = slope

        alpha_beta_results.append(
            [fund, alpha, beta]
        )

alpha_beta_df = pd.DataFrame(
    alpha_beta_results,
    columns=["amfi_code", "Alpha", "Beta"]
)

alpha_beta_df.head()

,amfi_code,Alpha,Beta
0,100016,0.037661,-0.000003
1,100025,0.044818,-0.000003
2,100033,0.265138,0.000012
3,101206,0.208464,0.000011
4,101207,0.109861,-0.000005


In [26]:
print(alpha_beta_df.head(10))

   amfi_code     Alpha          Beta
0     100016  0.037661 -3.399716e-06
1     100025  0.044818 -3.377072e-06
2     100033  0.265138  1.198524e-05
3     101206  0.208464  1.062703e-05
4     101207  0.109861 -4.983429e-06
5     101208  0.061229 -6.172390e-07
6     102885  0.153297  2.851745e-05
7     102886  0.031922 -7.302382e-06
8     102887  0.148102  2.496454e-05
9     118632  0.217238  1.373547e-06


In [27]:
drawdown_list = []

for code in nav["amfi_code"].unique():

    fund = nav[nav["amfi_code"] == code].copy()

    running_max = fund["nav"].cummax()

    drawdown = (fund["nav"] / running_max) - 1

    max_dd = drawdown.min()

    drawdown_list.append([code, max_dd])

drawdown_df = pd.DataFrame(
    drawdown_list,
    columns=["amfi_code", "Max_Drawdown"]
)

drawdown_df = drawdown_df.sort_values(
    "Max_Drawdown"
)

print(drawdown_df.head(10))

    amfi_code  Max_Drawdown
22     119599     -0.525742
17     119095     -0.516778
4      101207     -0.354469
39     149324     -0.311719
21     119598     -0.287060
7      102886     -0.280011
0      100016     -0.247344
29     120842     -0.240035
11     118634     -0.233449
15     119093     -0.217514


In [28]:
print(performance.columns)

Index(['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan',
       'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct',
       'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio',
       'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct',
       'morningstar_rating', 'risk_grade'],
      dtype='str')


In [29]:
scorecard = performance.copy()

scorecard["return_rank"] = scorecard["return_3yr_pct"].rank(
    ascending=False
)

scorecard["sharpe_rank"] = scorecard["sharpe_ratio"].rank(
    ascending=False
)

scorecard["alpha_rank"] = scorecard["alpha"].rank(
    ascending=False
)

scorecard["expense_rank"] = scorecard["expense_ratio_pct"].rank(
    ascending=True
)

scorecard["drawdown_rank"] = scorecard["max_drawdown_pct"].rank(
    ascending=True
)

scorecard["Fund_Score"] = (
    scorecard["return_rank"] * 0.30 +
    scorecard["sharpe_rank"] * 0.25 +
    scorecard["alpha_rank"] * 0.20 +
    scorecard["expense_rank"] * 0.15 +
    scorecard["drawdown_rank"] * 0.10
)

scorecard = scorecard.sort_values(
    "Fund_Score"
)

print(
    scorecard[
        ["amfi_code",
         "scheme_name",
         "Fund_Score"]
    ].head(10)
)

    amfi_code                                    scheme_name  Fund_Score
3      119599      SBI Small Cap Fund - Direct Plan - Growth      11.900
22     120843         Kotak Flexicap Fund - Regular - Growth      12.400
21     120842  Kotak Emerging Equity Fund - Regular - Growth      12.800
29     101207         ABSL Small Cap Fund - Regular - Growth      13.700
2      119598     SBI Small Cap Fund - Regular Plan - Growth      15.250
34     148567  Mirae Asset Large Cap Fund - Regular - Growth      15.425
9      100025   HDFC Short Term Debt Fund - Regular - Growth      15.800
14     120507       ICICI Pru Liquid Fund - Regular - Growth      16.300
12     120505       ICICI Pru Midcap Fund - Regular - Growth      16.600
11     120504      ICICI Pru Bluechip Fund - Direct - Growth      17.150


In [30]:
alpha_beta_df.to_csv(
    "../reports/alpha_beta.csv",
    index=False
)

print("alpha_beta.csv saved")

alpha_beta.csv saved


In [31]:
scorecard.to_csv(
    "../reports/fund_scorecard.csv",
    index=False
)

print("fund_scorecard.csv saved")

fund_scorecard.csv saved


In [32]:
print(benchmark["index_name"].unique())

<StringArray>
[        'NIFTY50',   'CRISIL_LIQUID', 'NIFTY_MIDCAP150',     'CRISIL_GILT',
        'NIFTY100',        'NIFTY500',    'BSE_SMALLCAP']
Length: 7, dtype: str


In [33]:
import plotly.express as px

top5 = scorecard.head(5)["amfi_code"].tolist()

top5_funds = performance[
    performance["amfi_code"].isin(top5)
][["amfi_code", "scheme_name", "return_3yr_pct"]]

fig = px.bar(
    top5_funds,
    x="scheme_name",
    y="return_3yr_pct",
    title="Top 5 Funds - 3 Year Return Comparison"
)

fig.show()

fig.write_html(
    "../reports/charts/benchmark_comparison.html"
)

In [34]:
performance[[
    "scheme_name",
    "return_3yr_pct",
    "benchmark_3yr_pct"
]].head()

,scheme_name,return_3yr_pct,benchmark_3yr_pct
0,SBI Bluechip Fund - Regular Plan - Growth,12.36,11.49
1,SBI Bluechip Fund - Direct Plan - Growth,11.30,9.52
2,SBI Small Cap Fund - Regular Plan - Growth,23.39,22.16
3,SBI Small Cap Fund - Direct Plan - Growth,23.14,22.01
4,SBI Magnum Gilt Fund - Regular Plan - Growth,6.07,4.47


In [35]:
import pandas as pd
import plotly.express as px

top5 = scorecard.head(5)

comparison_df = top5[
    ["scheme_name", "return_3yr_pct", "benchmark_3yr_pct"]
].copy()

comparison_long = comparison_df.melt(
    id_vars="scheme_name",
    value_vars=["return_3yr_pct", "benchmark_3yr_pct"],
    var_name="Type",
    value_name="Return (%)"
)

fig = px.bar(
    comparison_long,
    x="scheme_name",
    y="Return (%)",
    color="Type",
    barmode="group",
    title="Top 5 Funds vs Benchmark (3-Year Return)"
)

fig.show()

fig.write_html(
    "../reports/charts/benchmark_comparison.html"
)

In [36]:
top5 = performance.sort_values(
    "return_3yr_pct",
    ascending=False
).head(5)

print(top5[[
    "scheme_name",
    "return_3yr_pct",
    "benchmark_3yr_pct"
]])

                                       scheme_name  return_3yr_pct  \
2       SBI Small Cap Fund - Regular Plan - Growth           23.39   
3        SBI Small Cap Fund - Direct Plan - Growth           23.14   
29          ABSL Small Cap Fund - Regular - Growth           22.38   
27          Axis Small Cap Fund - Regular - Growth           20.98   
17  Nippon India Small Cap Fund - Regular - Growth           20.15   

    benchmark_3yr_pct  
2               22.16  
3               22.01  
29              20.54  
27              20.47  
17              19.35  


In [37]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_bar(
    x=top5["scheme_name"],
    y=top5["return_3yr_pct"],
    name="Fund Return"
)

fig.add_bar(
    x=top5["scheme_name"],
    y=top5["benchmark_3yr_pct"],
    name="Benchmark Return"
)

fig.update_layout(
    title="Top 5 Funds vs Benchmark (3 Year Return)",
    barmode="group"
)

fig.show()

fig.write_html(
    "../reports/charts/benchmark_comparison.html"
)

In [38]:
fig.update_layout(
    width=1400,
    height=700,
    xaxis_tickangle=-30,
    legend=dict(
        orientation="h",
        y=1.1,
        x=0.5,
        xanchor="center"
    )
)

In [39]:
import plotly.express as px

top10 = scorecard.sort_values(
    "Fund_Score",
    ascending=True
).head(10)

fig = px.bar(
    top10,
    x="Fund_Score",
    y="scheme_name",
    orientation="h",
    title="Top 10 Mutual Funds by Score",
    text="Fund_Score"
)

fig.update_layout(
    height=700
)

fig.show()

fig.write_html(
    "../reports/charts/top10_fund_scorecard.html"
)

In [40]:
fig.write_image(
    "../reports/charts/benchmark_comparison.png"
)

print("benchmark_comparison.png saved")

benchmark_comparison.png saved
